# Day 4: Multi-Object Tracking with YOLO and ByteTrack

Goal: run a YOLO detector on a short video, track multiple objects across frames with ByteTrack, and save a video showing stable track IDs.

I will use Ultralytics' built-in tracking mode with the ByteTrack tracker. The sample video downloads automatically. The output video is saved in the notebook runtime.

Install and import libraries

In [1]:
!pip -q install -U ultralytics

In [2]:
from pathlib import Path
import cv2
import torch
import matplotlib.pyplot as plt
from IPython.display import Video, display
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("OpenCV:", cv2.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
PyTorch: 2.14.1+cu130
OpenCV: 4.14.0
CUDA available: True
GPU: Tesla T4


Object tracking: what changes compared with detection?

Object detection works on an image or an individual video frame. It finds objects and predicts boxes, classes and confidence scores.

Multi-object tracking adds an identity over time:
- The detector finds objects in each frame.
- The tracker associates detections in the current frame with objects seen in earlier frames.
- Each tracked object receives a track ID, which should remain consistent while the same object is visible.

A detection ID is not the same as a track ID. A detection is a prediction for one frame; a track ID links detections across frames.

ByteTrack and DeepSORT

**ByteTrack** associates detections across frames using both high-confidence and lower-confidence detections. This can help maintain tracks when an object is temporarily detected with a lower score.

**DeepSORT** combines motion information with appearance features to associate objects. Appearance embeddings can help distinguish objects that look similar or cross paths, but they add computation.

In this notebook we use ByteTrack through Ultralytics' tracking API.

Download a short sample video

In [5]:
video_url = "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg"
video_path = Path("traffic.mp4")

import requests

response = requests.get(video_url, timeout=60)
response.raise_for_status()
video_path.write_bytes(response.content)

print("Video:", video_path.resolve())
print("File size (MB):", round(video_path.stat().st_size / (1024 * 1024), 2))

Video: /content/traffic.mp4
File size (MB): 0.13


Inspect the input video

In [6]:
cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError("Could not open the downloaded video.")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = frame_count / fps if fps else 0
cap.release()

print(f"Resolution: {width} x {height}")
print(f"FPS: {fps:.2f}")
print(f"Frames: {frame_count}")
print(f"Duration: {duration:.1f} seconds")
display(Video(str(video_path), embed=True, width=640))

Resolution: 810 x 1080
FPS: 25.00
Frames: -9223372036854775808
Duration: -368934881474191040.0 seconds


Load a pretrained YOLO model

In [7]:
# YOLO11n is a lightweight pretrained detector.
model = YOLO("yolo11n.pt")
print("YOLO11n loaded.")

YOLO11n loaded.


Run ByteTrack on the video

In [8]:
# ByteTrack is selected with tracker="bytetrack.yaml".
# Ultralytics draws each object's bounding box, class and track ID.
results = model.track(
    source=str(video_path),
    tracker="bytetrack.yaml",
    conf=0.25,
    iou=0.5,
    save=True,
    project="runs/module3",
    name="day4-bytetrack",
    exist_ok=True,
    stream=False,
    verbose=False
)

print("Frames processed:", len(results))
print("Tracking output directory:", model.predictor.save_dir)

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 2 packages in 432ms
Prepared 1 package in 125ms
Installed 1 package in 3ms
 + lap==0.5.14

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

Results saved to /content/runs/detect/runs/module3/day4-bytetrack
Frames processed: 1
Tracking output 

Inspect track IDs from the results

In [9]:
# Show a short summary from the first few frames.
for frame_index, result in enumerate(results[:10]):
    boxes = result.boxes
    if boxes is None or boxes.id is None:
        print(f"Frame {frame_index}: no tracked IDs returned")
        continue

    track_ids = boxes.id.int().cpu().tolist()
    class_ids = boxes.cls.int().cpu().tolist()
    labels = [model.names[class_id] for class_id in class_ids]
    print(f"Frame {frame_index}: IDs={track_ids}, classes={labels}")

Frame 0: IDs=[1, 2, 3, 4, 5], classes=['bus', 'person', 'person', 'person', 'person']


In [11]:
from pathlib import Path

print("Current working directory:", Path.cwd())

for p in Path("runs").rglob("*.mp4"):
    print(p, round(p.stat().st_size / (1024 * 1024), 2), "MB")

Current working directory: /content


Display the tracked output video

In [15]:
from pathlib import Path
from IPython.display import Video, display

output_dir = Path("/content/runs/detect/module3/day4-bytetrack")
video_files = list(output_dir.glob("*.avi")) + list(output_dir.glob("*.mp4"))

if video_files:
    output_video = video_files[0]
    print("Tracked video:", output_video.resolve())
    display(Video(str(output_video), embed=True, width=720))
else:
    print("No video file found.")

Tracked video: /content/runs/detect/module3/day4-bytetrack/traffic.avi


Understand the tracking output

Look at the IDs drawn beside the bounding boxes:

- The same ID appearing in nearby frames indicates the tracker associates those detections with the same object.
- A new ID may appear when a new object enters the scene.
- An ID can disappear when an object leaves the frame or is no longer detected.
- IDs may switch if the tracker loses an object and later associates it incorrectly.

Tracking quality depends on detector quality, frame rate, object movement, occlusion and tracker settings. A short visual check is useful, but it is not a formal tracking benchmark.

Save a copy of the tracked video

In [16]:
# Copy the generated video to a clear folder for the Module 3 repository.
import shutil

results_folder = Path("module3_day4_results")
results_folder.mkdir(exist_ok=True)
saved_video = results_folder / "bytetrack_output.mp4"
shutil.copy2(output_video, saved_video)

print("Saved tracked video:", saved_video.resolve())

Saved tracked video: /content/module3_day4_results/bytetrack_output.mp4
